# 追踪
跟踪是指应用程序从输入到输出所采取的一系列步骤。每个步骤都由一次运行表示。您可以使用LangSmith来可视化这些执行步骤。要使用它，请为您的应用程序启用跟踪功能。这使您可以执行以下操作：

- 调试本地运行的应用程序。
- 评估应用程序性能。
- 监视应用程序。

首先，在LangSmith注册一个免费帐户。

## 为您的应用程序启用跟踪¶
要为您的应用程序启用跟踪，请设置以下环境变量：

```shell
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=<your-api-key>
```

有关更多信息，请参阅使用 LangGraph 进行跟踪。

## 评估

要评估代理的性能，您可以使用LangSmith 评估函数。您需要首先定义一个评估函数来判断代理的结果，例如最终输出或轨迹。根据您的评估技术，这可能涉及或不涉及参考输出

In [ ]:
def evaluator(*, outputs: dict, reference_outputs: dict):
    # compare agent outputs against reference outputs
    output_messages = outputs["messages"]
    reference_messages = reference_outputs["messages"]
    score = compare_messages(output_messages, reference_messages)
    return {"key": "evaluator_score", "score": score}

首先，您可以使用`AgentEvals`包中预先构建的评估器：

`pip install -U agentevals`

## 创建评估器¶
评估代理性能的常用方法是将其轨迹（调用工具的顺序）与参考轨迹进行比较：

In [ ]:
import json
from agentevals.trajectory.match import create_trajectory_match_evaluator

outputs = [
    {
        "role": "assistant",
        "tool_calls": [
            {
                "function": {
                    "name": "get_weather",
                    "arguments": json.dumps({"city": "san francisco"}),
                }
            },
            {
                "function": {
                    "name": "get_directions",
                    "arguments": json.dumps({"destination": "presidio"}),
                }
            }
        ],
    }
]
reference_outputs = [
    {
        "role": "assistant",
        "tool_calls": [
            {
                "function": {
                    "name": "get_weather",
                    "arguments": json.dumps({"city": "san francisco"}),
                }
            },
        ],
    }
]

# Create the evaluator
evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="superset",  
)

# Run the evaluator
result = evaluator(
    outputs=outputs, reference_outputs=reference_outputs
)

下一步，了解有关如何自定义轨迹匹配评估器的更多信息。

## LLM¶
您可以使用 LLM-as-a-judge 评估器，它使用 LLM 将轨迹与参考输出进行比较并输出分数：

In [ ]:
import json
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE
)

evaluator = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
    model="openai:o3-mini"
)

## 运行评估器
要运行评估器，首先需要创建一个LangSmith 数据集。要使用预构建的 AgentEvals 评估器，您需要一个具有以下架构的数据集：
- 输入：`{"messages": [...]}`输入消息以调用代理。

- 输出：`{"messages": [...]}`代理输出中的预期消息历史记录。 对于轨迹评估，您可以选择仅保留助手消息。


In [ ]:
from langsmith import Client
from langgraph.prebuilt import create_react_agent
from agentevals.trajectory.match import create_trajectory_match_evaluator

client = Client()
agent = create_react_agent(...)
evaluator = create_trajectory_match_evaluator(...)

experiment_results = client.evaluate(
    lambda inputs: agent.invoke(inputs),
    # replace with your dataset name
    data="<Name of your dataset>",
    evaluators=[evaluator]
)

## 了解更多¶
- 在 LangSmith 中运行图表
- LangSmith 可观察性快速入门
- 使用 LangGraph 进行跟踪
- 追踪概念指南